<a href="https://colab.research.google.com/github/yasserjassem-H/SDN-sdn-rppo/blob/main/RPPOSDN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# 🔥 FULL SDN Recurrent PPO CODE FOR 5 PATHS (SC1 → SC5)
# ============================================================

# ==============================
# 1. Install Libraries
# ==============================
!pip install stable-baselines3[extra] sb3-contrib gymnasium pandas matplotlib torch scikit-learn seaborn shimmy>=2.0

# ==============================
# 2. Imports
# ==============================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

import gymnasium as gym
from gymnasium import spaces

from google.colab import files
import io

from sb3_contrib.ppo_recurrent import RecurrentPPO
from sb3_contrib.common.recurrent.policies import RecurrentActorCriticPolicy

from stable_baselines3.common.vec_env import DummyVecEnv
from stable_baselines3.common.callbacks import BaseCallback

# ============================================================
# 3. Upload CSV
# ============================================================

print("📂 Upload SDN CSV file")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
csv_data = uploaded[filename]

# ============================================================
# 4. QoS Function
# ============================================================

def calculate_qos_score(latency, bandwidth, pkt_loss, util):

    latency_weight = 7.5
    pkt_loss_weight = 3.0
    bandwidth_weight = 0.5
    util_weight = 3.0

    score = (
        latency * latency_weight
        + pkt_loss * pkt_loss_weight
        + util * util_weight
        - bandwidth * bandwidth_weight
    )

    return score

# ============================================================
# 5. Generate 5 Paths
# ============================================================

def load_and_prepare_sdn_data(file_content):

    df_raw = pd.read_csv(io.BytesIO(file_content))

    selected_cols = [
        'Latency',
        'bandwidth',
        'Pkt loss',
        'link_utilization'
    ]

    df = df_raw[selected_cols].copy()

    df.dropna(inplace=True)

    for col in selected_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df.dropna(inplace=True)

    np.random.seed(42)

    NUM_PATHS = 5

    # ========================================================
    # Create SC2 → SC5
    # ========================================================

    for path_id in range(2, NUM_PATHS + 1):

        df[f'latency_SC{path_id}'] = (
            df['Latency'] *
            np.random.uniform(0.5, 2.0, len(df))
        )

        df[f'bandwidth_SC{path_id}'] = (
            df['bandwidth'] *
            np.random.uniform(0.8, 1.2, len(df))
        )

        df[f'pkt_loss_SC{path_id}'] = (
            df['Pkt loss'] +
            np.random.uniform(-0.5, 0.5, len(df))
        ).clip(lower=0)

        df[f'util_SC{path_id}'] = (
            df['link_utilization'] *
            np.random.uniform(0.9, 1.1, len(df))
        )

    # ========================================================
    # Determine Optimal Action
    # ========================================================

    optimal_actions = []

    for _, row in df.iterrows():

        scores = []

        # ---------- SC1 ----------
        score_sc1 = calculate_qos_score(
            row['Latency'],
            row['bandwidth'],
            row['Pkt loss'],
            row['link_utilization']
        )

        scores.append(score_sc1)

        # ---------- SC2 → SC5 ----------
        for path_id in range(2, NUM_PATHS + 1):

            score = calculate_qos_score(
                row[f'latency_SC{path_id}'],
                row[f'bandwidth_SC{path_id}'],
                row[f'pkt_loss_SC{path_id}'],
                row[f'util_SC{path_id}']
            )

            scores.append(score)

        optimal_action = np.argmin(scores)

        optimal_actions.append(optimal_action)

    df['optimal_action'] = optimal_actions

    # ========================================================
    # Split Data
    # ========================================================

    df_train, df_test = train_test_split(
        df,
        test_size=0.2,
        random_state=42
    )

    return (
        df_train.reset_index(drop=True),
        df_test.reset_index(drop=True)
    )

# ============================================================
# Load Data
# ============================================================

df_train, df_test = load_and_prepare_sdn_data(csv_data)

print("✅ Train:", len(df_train))
print("✅ Test :", len(df_test))

# ============================================================
# 6. Gym Environment
# ============================================================

class SDNEnv(gym.Env):

    def __init__(self, df):

        super(SDNEnv, self).__init__()

        self.df = df.reset_index(drop=True)

        self.current_step = 0

        self.num_paths = 5

        # 4 metrics × 5 paths = 20 features
        self.observation_space = spaces.Box(
            low=0,
            high=1e9,
            shape=(20,),
            dtype=np.float32
        )

        # 5 actions
        self.action_space = spaces.Discrete(5)

    # ========================================================

    def reset(self, seed=None, options=None):

        super().reset(seed=seed)

        self.current_step = 0

        return self._get_obs(), {}

    # ========================================================

    def _get_obs(self):

        row_idx = min(self.current_step, len(self.df) - 1)

        row = self.df.iloc[row_idx]

        obs = [

            # SC1
            row['Latency'],
            row['bandwidth'],
            row['Pkt loss'],
            row['link_utilization']
        ]

        # SC2 → SC5
        for path_id in range(2, 6):

            obs.extend([

                row[f'latency_SC{path_id}'],
                row[f'bandwidth_SC{path_id}'],
                row[f'pkt_loss_SC{path_id}'],
                row[f'util_SC{path_id}']
            ])

        return np.array(obs, dtype=np.float32)

    # ========================================================

    def step(self, action):

        row = self.df.iloc[self.current_step]

        scores = []

        # ---------- SC1 ----------
        score_sc1 = calculate_qos_score(
            row['Latency'],
            row['bandwidth'],
            row['Pkt loss'],
            row['link_utilization']
        )

        scores.append(score_sc1)

        # ---------- SC2 → SC5 ----------
        for path_id in range(2, 6):

            score = calculate_qos_score(
                row[f'latency_SC{path_id}'],
                row[f'bandwidth_SC{path_id}'],
                row[f'pkt_loss_SC{path_id}'],
                row[f'util_SC{path_id}']
            )

            scores.append(score)

        optimal_action = np.argmin(scores)

        chosen_score = scores[action]

        optimal_score = scores[optimal_action]

        # ====================================================
        # Advanced Reward
        # ====================================================

        reward = 1.0 - (
            abs(chosen_score - optimal_score)
            /
            (abs(optimal_score) + 1e-6)
        )

        reward = np.clip(reward, -1, 1)

        self.current_step += 1

        terminated = self.current_step >= len(self.df)

        return (
            self._get_obs(),
            reward,
            terminated,
            False,
            {}
        )

# ============================================================
# 7. Callback
# ============================================================

class CustomCallback(BaseCallback):

    def __init__(
        self,
        eval_env,
        eval_df,
        threshold=0.985,
        eval_interval=5000
    ):

        super().__init__()

        self.eval_env = eval_env
        self.eval_df = eval_df

        self.threshold = threshold
        self.eval_interval = eval_interval

        self.accuracies = []

    # ========================================================

    def _on_step(self):

        if self.n_calls % self.eval_interval == 0:

            obs, _ = self.eval_env.reset()

            done = False

            hidden = None

            preds = []

            for _ in range(len(self.eval_df)):

                if done:
                    break

                action, hidden = self.model.predict(
                    obs,
                    state=hidden,
                    episode_start=np.array([done]),
                    deterministic=True
                )

                preds.append(int(action))

                obs, _, terminated, truncated, _ = \
                    self.eval_env.step(action.item())

                done = terminated or truncated

            acc = accuracy_score(
                self.eval_df['optimal_action'][:len(preds)],
                preds
            )

            self.accuracies.append((self.n_calls, acc))

            print(f"📊 Step {self.n_calls} | Accuracy: {acc:.4f}")

            if acc >= self.threshold:

                print("✅ Target Accuracy Reached")

                return False

        return True

# ============================================================
# 8. Training
# ============================================================

print("🚀 Creating environment")

venv_train = DummyVecEnv([lambda: SDNEnv(df_train)])

eval_env = SDNEnv(df_test)

# ============================================================

model = RecurrentPPO(

    policy=RecurrentActorCriticPolicy,

    env=venv_train,

    verbose=1,

    learning_rate=3e-4,

    n_steps=512,

    batch_size=1024,

    n_epochs=15,

    gamma=0.995,

    gae_lambda=0.98,

    ent_coef=0.003,

    clip_range=0.2,

    seed=42,

    tensorboard_log="./ppo_sdn_5paths/",

    policy_kwargs=dict(

        net_arch=dict(
            pi=[512, 512, 256],
            vf=[512, 512, 256]
        ),

        lstm_hidden_size=256
    )
)

# ============================================================

callback = CustomCallback(
    eval_env,
    df_test,
    threshold=0.985
)

# ============================================================

print("🏋️ Training Started")

model.learn(
    total_timesteps=1_500_000,
    callback=callback
)

# ============================================================

model.save("recurrent_ppo_sdn_5paths")

print("✅ Model Saved")

# ============================================================
# 9. Evaluation
# ============================================================

print("\n🔍 Final Evaluation")

obs, _ = eval_env.reset()

done = False

hidden = None

preds = []

for _ in range(len(df_test)):

    if done:
        break

    action, hidden = model.predict(
        obs,
        state=hidden,
        episode_start=np.array([done]),
        deterministic=True
    )

    preds.append(int(action))

    obs, _, terminated, truncated, _ = \
        eval_env.step(action.item())

    done = terminated or truncated

# ============================================================

accuracy = accuracy_score(
    df_test['optimal_action'][:len(preds)],
    preds
)

print(f"\n✅ FINAL ACCURACY = {accuracy:.4%}")

# ============================================================
# Confusion Matrix
# ============================================================

cm = confusion_matrix(
    df_test['optimal_action'][:len(preds)],
    preds
)

plt.figure(figsize=(10,8))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['SC1','SC2','SC3','SC4','SC5'],
    yticklabels=['SC1','SC2','SC3','SC4','SC5']
)

plt.title("Confusion Matrix - 5 Paths")

plt.xlabel("Predicted")

plt.ylabel("True")

plt.show()

# ============================================================
# Classification Report
# ============================================================

print("\n📝 Classification Report")

print(
    classification_report(
        df_test['optimal_action'][:len(preds)],
        preds,
        target_names=[
            'SC1',
            'SC2',
            'SC3',
            'SC4',
            'SC5'
        ]
    )
)

# ============================================================
# Accuracy Curve
# ============================================================

steps, accs = zip(*callback.accuracies)

plt.figure(figsize=(10,6))

plt.plot(
    steps,
    accs,
    marker='o'
)

plt.title("Accuracy During Training")

plt.xlabel("Steps")

plt.ylabel("Accuracy")

plt.grid(True)

plt.show()